# Milestone - Unit 4: hold 20 m/s, then follow the line

Unit 4 asked the submission to do two things the starter does not: hold its target speed, and follow the centreline.
`agents/unit-04.py` is Unit 3's file (its instruments and `predict` kept) with a small `PID` class and two controllers:

- speed: `PID(kp=0.5, ki=0.1, limits=(0, 1))` on `TARGET_SPEED - speed`, with conditional-integration anti-windup on the throttle's range, split into throttle and brake;
- steering: δ = −K_E·e + K_HEAD·ψ_e + atan(L·κ_i), with K_E = 0.2 rad/m and K_HEAD = 1.0: a PD law on the cross-track error whose D term is the measured heading error (K_HEAD = K_D·v), plus the curvature feedforward.

The notebook asserts, rather than displays, each fact the unit taught (plan roar-005, Gates U4.1–U4.4):

- **U4.1** the speed step response of the submission's own `PID`, on `KinematicCar()` (jitter 0, a straight, 10 substeps, 30 s): overshoot ±0.01 percentage points, settling time (2% band) exact to the tick, the error at 30 s ±1e-4 m/s;
- **U4.2** the submission at jitter 0: 216.35 / 400.55 / 858.45 s, all clean, RMS and max |offset| pinned; `check_submission` passes;
- **U4.3** the scorecard, five seeds (0–4) at jitter 0.05 on all three tracks, against the starter: the RMS median gap exceeds the larger range and Unit 4 is faster beyond the noise, all 30 races clean;
- **U4.4** the ablations: without the feedforward (five seeds, all three tracks) the RMS medians are worse than with it, and worse than the starter's on `test-circuit`; without the D term the car crashes (seed 0 here; the routed test `tests/test_racekit_unit04_05_premise.py` asserts the five-seed collision list).

Tolerances (the plan's rule): deterministic and median times ±0.05 s (one tick); RMS medians and ranges ±0.0005 m; max |offset| ±0.005 m; outcomes and collision counts exact; every comparison strict.
Every number is a number about racekit's **kinematic car**: a model, not the truth.

## 1. Setup

The milestone runs from `projects/racekit/milestones/`, so the submissions are one directory up.
U4.1 uses the submission's own `PID` class, taken from the loaded module.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit.cars import KinematicCar
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better, repeat_races, spread
from racekit.runner import CONTROL_DT, load_submission, run_race, shim_installed
from racekit.variants import write_variant

AGENT = Path("..") / "agents" / "unit-04.py"
STARTER = Path("..") / "agents" / "unit-01.py"
TRACKS = ("oval", "test-circuit", "monza-like")
TICK = 0.05 + 1e-9  # one control tick
RMS_TOL = 0.0005 + 1e-9
MAX_TOL = 0.005 + 1e-9

with shim_installed():
    _, module_name = load_submission(AGENT)
PID = sys.modules[module_name].PID  # the submission's own controller class


def rms_offset(result):
    offset = result.telemetry["offset"][result.telemetry["held"] == 0]
    return float(np.sqrt(np.mean(offset ** 2)))


def max_offset(result):
    offset = result.telemetry["offset"][result.telemetry["held"] == 0]
    return float(np.max(np.abs(offset)))


def jittered():
    return KinematicCar(jitter=0.05)

## 2. U4.1: the speed step response

From rest (and, for the PI rows, from 15 m/s) to 20 m/s on a straight, the controller sees the true speed every 0.05 s tick; the positive part of its command is the throttle and the negative part the brake.
Overshoot is a percentage of the step; the settling time is the first time from which the speed stays within 2% of the step of the target; the error is the target minus the speed at 30 s.
`PID(0.05)` is exactly the starter's speed loop.

In [ ]:
def step_response(controller, v0=0.0, target=20.0, seconds=30.0):
    car = KinematicCar()
    car.reset(0.0, 0.0, 0.0, v0)
    speeds = np.zeros(round(seconds / CONTROL_DT))
    for n in range(len(speeds)):
        command = controller.update(target - car.v, CONTROL_DT)
        car.advance(min(max(command, 0.0), 1.0), 0.0, min(max(-command, 0.0), 1.0), 0.0, 0, CONTROL_DT, 10)
        speeds[n] = car.v
    step = target - v0
    times = CONTROL_DT * np.arange(1, len(speeds) + 1)
    overshoot = max(0.0, float(np.max(speeds) - target) / step * 100.0)
    outside = np.nonzero(np.abs(speeds - target) > 0.02 * step)[0]
    if len(outside) == 0:
        settling = 0.0
    elif outside[-1] == len(speeds) - 1:
        settling = math.inf
    else:
        settling = float(times[outside[-1] + 1])
    return overshoot, settling, float(target - speeds[-1])


# controller -> (factory, v0, overshoot %, settling s (math.inf = never), error at 30 s or None)
U4_1 = {
    "P 0.05 (the starter)": (lambda: PID(0.05), 0.0, 0.0, math.inf, 0.5081),
    "P 0.5": (lambda: PID(0.5), 0.0, 0.0, 3.60, 0.0531),
    "P 2.0": (lambda: PID(2.0), 0.0, 0.0, 3.30, 0.0133),
    "PI (0.5, 0.1), no anti-windup": (lambda: PID(0.5, 0.1), 0.0, 28.71, 17.70, -0.0290),
    "PI (0.5, 0.1), anti-windup": (lambda: PID(0.5, 0.1, limits=(0.0, 1.0)), 0.0, 0.26, 3.55, -0.0003),
    "PI, no anti-windup, from 15 m/s": (lambda: PID(0.5, 0.1), 15.0, 7.19, 8.40, None),
    "PI, anti-windup, from 15 m/s": (lambda: PID(0.5, 0.1, limits=(0.0, 1.0)), 15.0, 0.98, 1.35, None),
}
for name, (factory, v0, overshoot, settling, error) in U4_1.items():
    measured = step_response(factory(), v0=v0)
    shown = "never" if measured[1] == math.inf else f"{measured[1]:.2f} s"
    print(f"{name:>32}: overshoot {measured[0]:6.2f}%, settling {shown:>7}, error at 30 s {measured[2]:+.4f} m/s")
    assert abs(measured[0] - overshoot) <= 0.01, (name, measured)
    if settling == math.inf:
        assert measured[1] == math.inf, (name, measured)
    else:
        assert abs(measured[1] - settling) < 1e-9, (name, measured)
    if error is not None:
        assert abs(measured[2] - error) <= 1e-4, (name, measured)

The P loop's error comes from the drag balance K_P·e·a_T = c·(v* − e)²: its smaller root is 0.5067 m/s for the starter (the 30 s figure is still settling), which is the starter's 19.49 m/s.

In [ ]:
A_THROTTLE, C_DRAG = 6.0, 4.0e-4
b = 2.0 * C_DRAG * 20.0 + A_THROTTLE * 0.05
equilibrium = (b - math.sqrt(b * b - 4.0 * C_DRAG * C_DRAG * 400.0)) / (2.0 * C_DRAG)
print(f"the starter's equilibrium error {equilibrium:.4f} m/s, speed {20.0 - equilibrium:.4f} m/s")
assert abs(equilibrium - 0.5067) < 1e-4 and abs(20.0 - equilibrium - 19.49) < 0.005

## 3. U4.2: the submission, deterministic

Jitter 0 on all three tracks: the pinned times, all clean, and the RMS and max |offset| over the driving rows.
Then `check_submission`: the file parses as Python 3.8, imports only what the official runner provides, and races `test-circuit` within the compute budget.

In [ ]:
U4_2 = {"oval": (216.35, 0.0034, 0.009), "test-circuit": (400.55, 0.0139, 0.136), "monza-like": (858.45, 0.0112, 0.153)}
for track, (time, rms, worst) in U4_2.items():
    result = run_race(AGENT, track, car=KinematicCar(jitter=0.0))
    print(f"{result}; RMS offset {rms_offset(result):.4f} m, max {max_offset(result):.3f} m")
    assert result.finished and result.collisions == 0, result
    assert abs(result.total_time - time) <= TICK, result
    assert abs(rms_offset(result) - rms) <= RMS_TOL and abs(max_offset(result) - worst) <= MAX_TOL, track

report = check_submission(AGENT)
print(report)
assert report.ok, report.findings

## 4. U4.3: the scorecard against the starter

Five seeds (0–4) at jitter 0.05 on each track, for both submissions (the starter keeps its P speed loop).
The pins are (RMS median, RMS range, time median, time spread); the spread is the range, max − min (Unit 1).

In [ ]:
SCORECARD = {
    "unit 4": {"oval": (0.1041, 0.0062, 216.40, 0.05), "test-circuit": (0.1048, 0.0050, 400.60, 0.10),
               "monza-like": (0.1029, 0.0042, 858.55, 0.00)},
    "starter": {"oval": (0.1397, 0.0099, 220.75, 0.35), "test-circuit": (0.1586, 0.0055, 407.50, 0.65),
                "monza-like": (0.1416, 0.0040, 870.50, 0.45)},
}
RUNS = {}
for track in TRACKS:
    for name, submission in (("unit 4", AGENT), ("starter", STARTER)):
        runs = repeat_races(submission, track, n=5, car_factory=jittered)
        rms = [rms_offset(result) for result in runs.results]
        RUNS[(name, track)] = (runs, rms)
        rms_median, rms_range, time_median, time_spread = SCORECARD[name][track]
        print(f"{track:>12} {name:>8}: RMS median {np.median(rms):.4f} (range {spread(rms):.4f}), "
              f"time median {runs.median:.2f} s (spread {runs.spread:.2f}), collisions {runs.collisions}")
        assert runs.all_finished and runs.collisions == 0, (name, track, runs)
        assert abs(np.median(rms) - rms_median) <= RMS_TOL and abs(spread(rms) - rms_range) <= RMS_TOL, (name, track)
        assert abs(runs.median - time_median) <= TICK and abs(runs.spread - time_spread) <= TICK, (name, track, runs)
    (u4_runs, u4_rms), (st_runs, st_rms) = RUNS[("unit 4", track)], RUNS[("starter", track)]
    assert np.median(st_rms) - np.median(u4_rms) > max(spread(u4_rms), spread(st_rms)), track
    assert noise_aware_better(u4_runs.times, st_runs.times), track

## 5. U4.4: the ablations

`write_variant` turns each ablation into a real submission.
Without the feedforward (`FEEDFORWARD = False`) the car must hold every corner with an error, so the RMS median is worse than with it on all three tracks, and worse than the starter's on `test-circuit`; every race is still clean.
Without the D term (`K_HEAD = 0.0`, `max_seconds=400`) the car weaves off the track and never finishes.

In [ ]:
folder = Path(mkdtemp())
no_feedforward = write_variant(AGENT, folder / "no_feedforward.py", {"FEEDFORWARD": "False"})
NO_FEEDFORWARD = {"oval": 0.1327, "test-circuit": 0.1794, "monza-like": 0.1374}
for track, pinned in NO_FEEDFORWARD.items():
    runs = repeat_races(no_feedforward, track, n=5, car_factory=jittered)
    rms = [rms_offset(result) for result in runs.results]
    print(f"{track:>12} no feedforward: RMS median {np.median(rms):.4f} m (with it {np.median(RUNS[('unit 4', track)][1]):.4f}), "
          f"collisions {runs.collisions}")
    assert runs.all_finished and runs.collisions == 0, (track, runs)
    assert abs(np.median(rms) - pinned) <= RMS_TOL, (track, rms)
    assert np.median(rms) > np.median(RUNS[("unit 4", track)][1]), track
    if track == "test-circuit":
        assert np.median(rms) > np.median(RUNS[("starter", track)][1])

no_d_term = write_variant(AGENT, folder / "no_d_term.py", {"K_HEAD": "0.0"})
crash = run_race(no_d_term, "test-circuit", car=jittered(), seed=0, max_seconds=400.0)
print(f"no D term, seed 0: {crash}")
assert not crash.finished and crash.collisions == 30, crash

## Unit 4, pinned

| | starter | Unit 4 |
|---|---|---|
| speed step from rest | settles never, 0.51 m/s short | 0.26% overshoot, 3.55 s, no error (PI with anti-windup) |
| jitter 0 race times | 223.50 / 412.75 / 882.55 s | 216.35 / 400.55 / 858.45 s |
| RMS offset, five-seed medians | 0.1397 / 0.1586 / 0.1416 m | 0.1041 / 0.1048 / 0.1029 m |
| time, five-seed medians | 220.75 / 407.50 / 870.50 s | 216.40 / 400.60 / 858.55 s |

Every term earns its place: without the I term the speed falls short, without anti-windup it overshoots by 29%, without the D term the car crashes, and without the feedforward it holds the corners off the line.